# 🧪 Cuaderno 02 — Laboratorios y Prácticas de Métodos

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Laboratorios interactivos y preguntas sobre los **4 métodos**. La **teoría completa**
está en los markdown (enlazados en cada sección); aquí solo vienes a **practicar y
comprobar** que entiendes.

> 📖 **Teoría (markdown):** [Eliminación Gaussiana](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/01_eliminacion_gaussiana.md) · [LU](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/02_descomposicion_lu.md) · [Gauss-Seidel](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/03_metodos_iterativos.md) · [Thomas/Cholesky](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/04_matrices_especiales.md)


In [ ]:
#@title ⚙️ Cargar entorno (ejecuta PRIMERO)
import os, urllib.request
try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

ARCHIVO = "robomat.py"
if ES_COLAB and not os.path.exists(ARCHIVO):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/<USUARIO>/<REPO>/main/lib/robomat.py", ARCHIVO)

exec(open(ARCHIVO, encoding="utf-8").read(), globals())
import numpy as np
from ipywidgets import interact, IntSlider, FloatSlider
print("✅ Entorno cargado")


## 1. Eliminación Gaussiana · laboratorio interactivo

Mueve el deslizador para ver la matriz aumentada tras cada **pivote** (eliminación hacia adelante).
📖 [Teoría completa](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/01_eliminacion_gaussiana.md)


In [ ]:
A = np.array([[3., -0.1, -0.2], [0.1, 7., -0.3], [0.3, -0.2, 10.]])
b = np.array([7.85, -19.3, 71.4])

def matriz_tras_pivote(k):
    Ab = np.c_[A.copy(), b.copy()]
    n = len(b)
    for j in range(min(k, n-1)):
        p = j + np.argmax(np.abs(Ab[j:, j]))
        if p != j: Ab[[j, p]] = Ab[[p, j]]
        for i in range(j+1, n):
            m = Ab[i, j] / Ab[j, j]
            Ab[i, j:] = Ab[i, j:] - m * Ab[j, j:]
    print(Ab)

interact(matriz_tras_pivote, k=IntSlider(min=0, max=2, step=1, value=0, description="pivote"))


### ✏️ Pregunta 1 (teórica)
¿Qué problema(s) resuelve el **pivoteo parcial**?


In [ ]:
opcion = "b"  #@param ["a: acelera la convergencia", "b: evita división entre cero y amplificación de error", "c: reduce la memoria", "d: no resuelve nada"]
CORRECTA = "b"
if opcion.startswith(CORRECTA):
    print("✅ Correcto: el pivoteo evita pivote nulo y minimiza el error de redondeo.")
else:
    print("❌ Revisa la sección de pivoteo en el markdown.")


### 🔢 Pregunta 2 (numérica)
Resuelve a mano este sistema $2\times2$ y escribe $x_1$:

$$\begin{bmatrix} 2 & 1 \\ 1 & 3 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} = \begin{bmatrix} 4 \\ 7 \end{bmatrix}$$


In [ ]:
x1 = 1.0  #@param {type:"number"}
correcto = np.linalg.solve(np.array([[2.,1.],[1.,3.]]), np.array([4.,7.]))[0]
if abs(x1 - correcto) < 1e-6:
    print(f"✅ Correcto, x1 = {correcto}")
else:
    print(f"❌ Incorrecto. Pista: despeja x2 de la 1ª y sustituye en la 2ª.")


## 2. Descomposición LU · laboratorio interactivo

Factoriza una matriz y verifica $\mathbf{A}=\mathbf{L}\mathbf{U}$.
📖 [Teoría completa](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/02_descomposicion_lu.md)


In [ ]:
def descomponer_lu(A):
    A = np.array(A, dtype=float).copy(); n = A.shape[0]
    L = np.eye(n); U = np.zeros((n,n))
    for k in range(n):
        U[k, k:] = A[k, k:]
        for i in range(k+1, n):
            L[i,k] = A[i,k]/U[k,k]
            A[i,k:] = A[i,k:] - L[i,k]*U[k,k:]
    return L, U

L, U = descomponer_lu(np.array([[3.,-0.1,-0.2],[0.1,7.,-0.3],[0.3,-0.2,10.]]))
print("L =\n", L)
print("U =\n", U)
print("¿A == L@U?", np.allclose(np.array([[3.,-0.1,-0.2],[0.1,7.,-0.3],[0.3,-0.2,10.]]), L@U))


### ✏️ Pregunta 3 (teórica)
¿Cuándo conviene usar **LU** en lugar de eliminación gaussiana simple?


In [ ]:
opcion = "b"  #@param ["a: siempre, es más rápido", "b: cuando hay que resolver varios vectores b con la misma A", "c: solo para matrices tridiagonales", "d: nunca"]
if opcion.startswith("b"):
    print("✅ Correcto: factorizas una vez y reutilizas para muchos b.")
else:
    print("❌ Revisa la motivación de LU en el markdown.")


## 3. Gauss-Seidel · convergencia interactiva

Mueve el deslizador para ver cómo converge la solución. 📖 [Teoría completa](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/03_metodos_iterativos.md)


In [ ]:
def gs_hist(iteraciones):
    A = np.array([[3.,-0.1,-0.2],[0.1,7.,-0.3],[0.3,-0.2,10.]])
    b = np.array([7.85,-19.3,71.4]); n = len(b)
    x = np.zeros(n); xref = np.linalg.solve(A,b)
    err = []
    for _ in range(iteraciones):
        xo = x.copy()
        for i in range(n):
            s = np.dot(A[i,:i],x[:i]) + np.dot(A[i,i+1:],xo[i+1:])
            x[i] = (b[i]-s)/A[i,i]
        err.append(np.max(np.abs(x-xref)))
    fig, ax = plt.subplots()
    ax.semilogy(range(1, iteraciones+1), err, marker="o")
    ax.set_xlabel("iteración"); ax.set_ylabel("error (log)"); ax.set_title("Convergencia de Gauss-Seidel")
    plt.show()

interact(gs_hist, iteraciones=IntSlider(min=1, max=15, step=1, value=1))


### ✏️ Pregunta 4 (teórica)
¿Qué condición **garantiza** la convergencia de Gauss-Seidel?


In [ ]:
opcion = "c"  #@param ["a: matriz simétrica", "b: matriz densa", "c: dominancia diagonal estricta", "d: matriz con ceros en la diagonal"]
if opcion.startswith("c"):
    print("✅ Correcto.")
else:
    print("❌ Revisa la condición de dominancia diagonal.")


## 4. Algoritmo de Thomas · laboratorio

Resuelve un sistema tridiagonal en $O(n)$.
📖 [Teoría completa](https://github.com/<USUARIO>/<REPO>/blob/main/docs/metodos/04_matrices_especiales.md)


In [ ]:
def thomas(e, f, g, r):
    n = len(f); e=np.array(e,float); f=np.array(f,float).copy()
    g=np.array(g,float); r=np.array(r,float).copy()
    for k in range(1,n):
        fac=e[k]/f[k-1]; f[k]-=fac*g[k-1]; r[k]-=fac*r[k-1]
    x=np.zeros(n); x[-1]=r[-1]/f[-1]
    for k in range(n-2,-1,-1): x[k]=(r[k]-g[k]*x[k+1])/f[k]
    return x

f=[2.,2.,2.]; e=[0.,-1.,-1.]; g=[-1.,-1.,0.]; r=[1.,0.,1.]
print("Solución (Thomas):", thomas(e,f,g,r))


### ✏️ Pregunta 5 (teórica)
¿Cuál es la **complejidad** del algoritmo de Thomas para un sistema $n\times n$ tridiagonal?


In [ ]:
opcion = "a"  #@param ["a: O(n)", "b: O(n²)", "c: O(n³)", "d: O(log n)"]
if opcion.startswith("a"):
    print("✅ Correcto: O(n), mucho más rápido que O(n³) de Gauss.")
else:
    print("❌ Revisa la complejidad en el markdown.")


## 5. Python vs MATLAB (comparación teórica)

| Operación | Python (NumPy/SciPy) | MATLAB |
|---|---|---|
| Resolver $\mathbf{A}\mathbf{x}=\mathbf{b}$ | `np.linalg.solve(A, b)` | `A \\ b` |
| Descomposición LU | `scipy.linalg.lu(A)` | `[L,U,P] = lu(A)` |
| Determinante / inversa | `np.linalg.det` / `inv` | `det(A)` / `inv(A)` |
| Número de condición | `np.linalg.cond(A)` | `cond(A)` |
| Factorización Cholesky | `np.linalg.cholesky(A)` | `chol(A)` |

**En síntesis:** la **barra invertida** de MATLAB (`A \\ b`) elige automáticamente el mejor solver,
mientras que en Python debes elegir la función. Ambos usan las mismas rutinas LAPACK/BLAS por debajo.


## 6. Resumen de tu práctica

Revisa tus respuestas arriba (✅/❌). El **envío a Google Sheets** vía Apps Script
se habilita en la **Fase 4** del repositorio.
